## Collaborative filtering

This notebook contains the experiments and analyses done for implementing collaborative filtering into the recommender system.

In [6]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

reviews_dir = PROJECT_ROOT / "data" / "raw" / "recommendations.csv"
df = pd.read_csv(reviews_dir)

users_repeated = (df['user_id'].value_counts() > 1).sum()
games_repeated = (df['app_id'].value_counts() > 1).sum()

print("Number of users repeated (Before) :", users_repeated)
print("Number of games repeated (Before) :", games_repeated)
print("Total interactions:", len(df))

Number of users repeated (Before) : 6208032
Number of games repeated (Before) : 37319
Total interactions: 41154794


There are a lot of interactions in the dataset that are noise. Removing users with fewer than 5 interactions and games with fewer than 10 reviews will improve the quality of the dataset and, therefore, the model significantly.

In [7]:
user_counts = df['user_id'].value_counts()
df = df[df['user_id'].isin(user_counts[user_counts >= 5].index)]

game_counts = df['app_id'].value_counts()
df = df[df['app_id'].isin(game_counts[game_counts >= 100].index)]

users_repeated = (df['user_id'].value_counts() > 1).sum()
games_repeated = (df['app_id'].value_counts() > 1).sum()

print("Number of users repeated (After) :", users_repeated)
print("Number of games repeated (After) :", games_repeated)
print("Total interactions:", len(df))

Number of users repeated (After) : 1910488
Number of games repeated (After) : 11049
Total interactions: 21772127


In [8]:
num_users = df['user_id'].nunique()
num_games = df['app_id'].nunique()
num_interactions = len(df)

sparsity = 1 - (num_interactions / (num_users * num_games)) # Shows % of the matrix (user-game combinations) that is empty

print("Sparsity:", sparsity )

Sparsity: 0.9989687223443028


Check for missing values on key columns and duplicate interactions

In [9]:
print(df[['user_id', 'app_id', 'is_recommended', 'hours']].isnull().sum())

duplicates = df.duplicated(subset=['user_id', 'app_id']).sum()
print("Duplicate user-game pairs:", duplicates)
df = df.drop_duplicates(subset=['user_id', 'app_id'])

print("Total interactions after removing duplicates and missing values:", len(df))

user_id           0
app_id            0
is_recommended    0
hours             0
dtype: int64
Duplicate user-game pairs: 19
Total interactions after removing duplicates and missing values: 21772108


In [10]:
print(df['is_recommended'].value_counts(normalize=True))
print(df['hours'].describe())
print("Users with 0 hours:", (df['hours'] == 0).sum())
df = df[df['hours'] > 0]
print("Total interactions after removing zero-hour reviews:", len(df))

is_recommended
True     0.850163
False    0.149837
Name: proportion, dtype: float64
count    2.177211e+07
mean     7.745022e+01
std      1.491146e+02
min      0.000000e+00
25%      6.200000e+00
50%      2.050000e+01
75%      6.950000e+01
max      1.000000e+03
Name: hours, dtype: float64
Users with 0 hours: 103018
Total interactions after removing zero-hour reviews: 21669090


In [11]:
top10_users = df['user_id'].value_counts().head(int(num_users * 0.1)).sum()
print(f"Top 10% users account for {top10_users / num_interactions * 100:.1f}% of interactions")

top10_games = df['app_id'].value_counts().head(int(num_games * 0.1)).sum()
print(f"Top 10% games account for {top10_games / num_interactions * 100:.1f}% of interactions")

Top 10% users account for 34.7% of interactions
Top 10% games account for 72.0% of interactions


In [15]:
import numpy as np

df['rating'] = df['is_recommended'].astype(int)

# Cap outliers at the 99th percentile and apply log transformation to normalize skewed 'hours' data
df['hours_log'] = np.log(df['hours'].clip(upper=df['hours'].quantile(0.99)))

df_cf = df[['user_id', 'app_id', 'rating', 'hours_log']].copy()

print(df_cf.head())

    user_id   app_id  rating  hours_log
0     51580   975370       1   3.591818
12   161081  1938090       1   3.843744
16    76583   635260       1   5.176150
21   664486  1938090       1   2.001480
22    22793   534380       1   3.703768
